# WA active-school features for the 49 model sites

## About the data source

This notebook uses the WA Department of Education **Current Active Schools Semester 1 2026 (DET-024)** workbook. It is a point dataset in WGS84 coordinates with one row per active school. The fixed 2026 Semester 1 snapshot contains 1,152 schools and 19 source columns. Despite the word “Public” in the catalogue title, the workbook contains `GOVERNMENT`, `NON-GOVERNMENT` and `COMMUNITY KINDER` schools, so this notebook describes it as an **active-schools dataset**, not a public-schools-only dataset.

**Access method / API:** the 2026 DET-024 catalogue package currently provides no web-service API. This notebook therefore does not call an education API. It uses the official Data WA XLSX resource download URL over HTTPS and stores the file as a fixed local snapshot at `data/external_raw/wa_education/2026_S1/currentactiveschools20261.xlsx`. Python's standard `urllib.request` downloads the workbook only when the local snapshot is missing; all feature extraction then reads that local file. The older 2025 ArcGIS REST layer is not used because it is a different, outdated annual dataset.

The source tells us where each school is and supplies its school code, name, physical address, education region, sector, remoteness classification, lowest and highest year codes, and reported primary, secondary and total student counts. This information can support measures of school proximity, the type of the nearest school and the year range it serves. It cannot tell us the walking route, school gate location, student travel mode, pedestrian volume, whether students cross the surveyed road, or the amount of school-related pedestrian exposure at that site.

## What this notebook produces

The notebook reads the 49 survey sites from `data/processed/model_dataset.xlsx`, uses each site's WGS84 coordinate to find the nearest active school within the declared Western Australia project extent, and calculates a straight-line planar distance in the site's local UTM coordinate reference system. It writes one independent row per `site_id` to `haoqi-05-wa-education-features.xlsx`; it does not copy model targets, coordinates or unrelated fields into the feature table.

| Output column | Clear meaning | Modelling role |
|---|---|---|
| `site_id` | Stable survey-site key used to join this table back to `Model_Data`. | Join key only; not a predictor. |
| `education_nearest_active_school_distance_m` | Straight-line planar distance in metres from the survey site to the nearest in-scope active-school point. | The only recommended education predictor in the first model. |
| `education_nearest_school_sector` | Provider-supplied sector of the nearest school: government, non-government or community kinder. | Interpretation and pre-declared sensitivity analysis; not a default first-model predictor. |
| `education_nearest_school_low_year` | Provider-supplied lowest year code of the nearest school, such as `KIN`, `PKG` or `Y07`. | Interpretation and sensitivity analysis; preserve as a category. |
| `education_nearest_school_high_year` | Provider-supplied highest year code of the nearest school, such as `Y06`, `Y10` or `Y12`. | Interpretation and sensitivity analysis; preserve as a category. |
| `education_nearest_school_code` | Official identifier of the matched nearest school. | Audit field only; never a predictor. |
| `education_nearest_school_name` | Official name of the matched nearest school. | Audit and manual checking only; never a predictor. |
| `education_data_status` | Whether the site was successfully matched, or whether a coordinate/source problem occurred. | Data-quality field only; never a substantive predictor. |

The first model should therefore use only the raw distance column from this source. School sector and year range remain available for a separately declared sensitivity check. School identifiers and match status are retained so every result can be traced and checked. This is feature extraction, not model fitting; alternative definitions or transformations must be evaluated inside the cross-validation workflow rather than selected after viewing all targets.

In [1]:
from hashlib import sha256
import os
from pathlib import Path
import tempfile
from urllib.request import Request, urlopen

matplotlib_cache = Path(tempfile.gettempdir()) / "cits5553-matplotlib-cache"
matplotlib_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(matplotlib_cache))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyproj import Transformer


def find_project_root() -> Path:
    # Find the project repository root from a notebook or repository working directory.
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (
            (path / "requirements.txt").is_file()
            and (path / "data" / "processed" / "model_dataset.xlsx").is_file()
        ):
            return path
    raise FileNotFoundError(
        f"Could not find the project repository above {current}. Run the notebook inside the project workspace."
    )


PROJECT_ROOT = find_project_root()
MODEL_DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "model_dataset.xlsx"
SCHOOL_SOURCE_PATH = (
    PROJECT_ROOT
    / "data"
    / "external_raw"
    / "wa_education"
    / "2026_S1"
    / "currentactiveschools20261.xlsx"
)
OUTPUT_DIR = (
    PROJECT_ROOT / "outputs" / "02_imagery_features" / "wa-education-schools"
)
OUTPUT_PATH = OUTPUT_DIR / "haoqi-05-wa-education-features.xlsx"

CATALOGUE_URL = (
    "https://catalogue.data.wa.gov.au/dataset/"
    "current-active-schools-semester-1-2026-public-det-024"
)
DOWNLOAD_URL = (
    "https://catalogue.data.wa.gov.au/dataset/"
    "90b5a308-edd6-4814-89f7-1af4df14687b/resource/"
    "d1bec5f9-e284-4484-af4d-cc7a69a55c5a/download/"
    "currentactiveschools20261.xlsx"
)


def download_fixed_snapshot(url: str, destination: Path) -> None:
    # Download the fixed source snapshot only when it is not already stored locally.
    if destination.is_file():
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    request = Request(url, headers={"User-Agent": "CITS5553-capstone/1.0"})
    with urlopen(request, timeout=120) as response:
        content = response.read()
    if len(content) < 10_000:
        raise ValueError("The downloaded school workbook is unexpectedly small.")
    destination.write_bytes(content)


download_fixed_snapshot(DOWNLOAD_URL, SCHOOL_SOURCE_PATH)
source_hash = sha256(SCHOOL_SOURCE_PATH.read_bytes()).hexdigest()
print(f"Project root: {PROJECT_ROOT}")
print(f"School snapshot: {SCHOOL_SOURCE_PATH}")
print(f"SHA-256: {source_hash}")

Project root: /workspace
School snapshot: /workspace/data/external_raw/wa_education/2026_S1/currentactiveschools20261.xlsx
SHA-256: ac4257d76e333dcbf2042c7e89b6337fd66b4a990667bc936ed2b1172dd2ffbd


## 1. Read the authoritative model-site input

The `Model_Data` sheet is the foundation table for model development. Its current contract defines `site_id` as the stable survey-site key, `latitude` and `longitude` as WGS84 decimal-degree coordinates, `coordinate_crs` as the recorded CRS, and `qa_status` as the row-level approval status.

Feature extraction covers all 49 sites so that feature availability is independent of later training eligibility. Model training can subsequently use `qa_status == "ready"`; the education output itself does not repeat `qa_status`, coordinates, road labels or observed targets. This separation prevents a source-specific feature table from becoming a renamed copy of `model_dataset.xlsx`.

In [2]:
model_data = pd.read_excel(
    MODEL_DATASET_PATH,
    sheet_name="Model_Data",
    dtype={"site_id": "string", "coordinate_crs": "string", "qa_status": "string"},
)

site_columns = ["site_id", "latitude", "longitude", "coordinate_crs", "qa_status"]
missing_site_columns = set(site_columns) - set(model_data.columns)
if missing_site_columns:
    raise ValueError(f"Model_Data is missing required columns: {sorted(missing_site_columns)}")

sites = model_data[site_columns].copy()
sites["site_id"] = sites["site_id"].str.strip()
sites["coordinate_crs"] = sites["coordinate_crs"].str.strip().str.upper()
sites["qa_status"] = sites["qa_status"].str.strip().str.lower()

if len(sites) != 49:
    raise ValueError(f"Expected the current 49-site model dataset, found {len(sites)} rows.")
if sites["site_id"].isna().any() or sites["site_id"].duplicated().any():
    raise ValueError("site_id must be complete and unique before feature extraction.")
if sites[["latitude", "longitude"]].isna().any().any():
    raise ValueError("All model sites must have coordinates before school matching.")
if not sites["latitude"].between(-90, 90).all():
    raise ValueError("At least one site latitude is outside the valid WGS84 range.")
if not sites["longitude"].between(-180, 180).all():
    raise ValueError("At least one site longitude is outside the valid WGS84 range.")
if set(sites["coordinate_crs"].dropna()) != {"EPSG:4326"}:
    raise ValueError("This workflow requires every model-site coordinate to be EPSG:4326.")

print(f"Model sites: {len(sites)}")
print(f"Unique site_id values: {sites['site_id'].nunique()}")
print("QA status distribution:")
print(sites["qa_status"].value_counts(dropna=False).to_string())

Model sites: 49
Unique site_id values: 49
QA status distribution:
qa_status
ready    49


## 2. Read and verify the official 2026 school snapshot

The official [Data WA catalogue record](https://catalogue.data.wa.gov.au/dataset/current-active-schools-semester-1-2026-public-det-024) says that DET-024 shows current active school locations for Semester 1 of the 2026 school year. It was published by the WA Department of Education, last updated on 13 May 2026, uses GCS WGS 1984 coordinates, and is licensed under [Creative Commons Attribution-NonCommercial 4.0](https://creativecommons.org/licenses/by-nc/4.0/). Any client-facing reuse must retain attribution and confirm that the intended delivery is compatible with the NonCommercial licence.

The catalogue does not provide a separate field dictionary, so the notebook does not invent stronger definitions than the published workbook supports. It uses the provider's field labels conservatively: `Code` and `SchoolName` identify the school; `Latitude` and `Longitude` locate it; `Sector` is the provider's sector classification; and `LowYear`/`HighYear` are the provider's lower and upper year codes. Although the package title contains “Public”, the workbook itself contains `GOVERNMENT`, `NON-GOVERNMENT` and `COMMUNITY KINDER` records. The analysis therefore calls these **active schools** and preserves the supplied `Sector` values.

In [3]:
school_workbook = pd.ExcelFile(SCHOOL_SOURCE_PATH)
if school_workbook.sheet_names != ["ActiveSchools"]:
    raise ValueError(
        f"Unexpected school workbook sheets: {school_workbook.sheet_names}. "
        "Expected only ActiveSchools."
    )

school_columns = [
    "Code",
    "SchoolName",
    "PhysicalTown",
    "Latitude",
    "Longitude",
    "EducationRegionName",
    "Sector",
    "RemCode",
    "RemoteArea",
    "LowYear",
    "HighYear",
    "TotalPrimaryK_6",
    "TotalSecondary7_12",
    "TotalSchool",
]
schools = pd.read_excel(
    SCHOOL_SOURCE_PATH,
    sheet_name="ActiveSchools",
    dtype={"Code": "string"},
)
missing_school_columns = set(school_columns) - set(schools.columns)
if missing_school_columns:
    raise ValueError(
        f"ActiveSchools is missing required fields: {sorted(missing_school_columns)}"
    )

schools["Code"] = schools["Code"].str.strip()
for column in ["SchoolName", "PhysicalTown", "Sector", "LowYear", "HighYear"]:
    schools[column] = schools[column].astype("string").str.strip()

if len(schools) != 1_152:
    raise ValueError(f"Expected 1,152 rows in the fixed 2026 snapshot, found {len(schools)}.")
if schools[school_columns].isna().any().any():
    raise ValueError("The selected official school fields unexpectedly contain missing values.")
if schools["Code"].duplicated().any():
    raise ValueError("School Code must be unique in the fixed snapshot.")
if not schools["Latitude"].between(-90, 90).all():
    raise ValueError("At least one school latitude is invalid.")
if not schools["Longitude"].between(-180, 180).all():
    raise ValueError("At least one school longitude is invalid.")

print(f"Official source rows: {len(schools):,}")
print(f"Unique school codes: {schools['Code'].nunique():,}")
print("Sector distribution:")
print(schools["Sector"].value_counts(dropna=False).to_string())
print("Low-year values:", sorted(schools["LowYear"].unique().tolist()))
print("High-year values:", sorted(schools["HighYear"].unique().tolist()))
print(
    "Zero TotalSchool records:",
    int(schools["TotalSchool"].eq(0).sum()),
)

Official source rows: 1,152
Unique school codes: 1,152
Sector distribution:
Sector
GOVERNMENT          817
NON-GOVERNMENT      317
COMMUNITY KINDER     18
Low-year values: ['KIN', 'PKG', 'PRP', 'Y01', 'Y03', 'Y04', 'Y05', 'Y07', 'Y08', 'Y09', 'Y10', 'Y11']
High-year values: ['KIN', 'Y02', 'Y03', 'Y04', 'Y06', 'Y07', 'Y09', 'Y10', 'Y11', 'Y12']
Zero TotalSchool records: 2


## 3. Define the spatial match and the feature roles before using targets

The source contains two WA-administered schools on Christmas Island and the Cocos (Keeling) Islands. They are outside this road project's spatial domain. Rather than deleting names manually, the code applies an explicit project extent—longitude 112°E to 129°E and latitude 36°S to 13°S—which contains all 49 model sites and mainland WA project locations. It then checks which fixed-snapshot records fell outside that rule.

For each site, the code estimates its southern-hemisphere UTM zone from longitude, projects the site and all in-scope schools into that local CRS, and selects the minimum Euclidean distance. The result is a planar straight-line distance between two points. It is not walking-network distance, school-gate distance, a claim that students cross the surveyed road, or a measure of school quality.

The output deliberately separates modelling and audit roles:

| Output field | Role and decision |
|---|---|
| `education_nearest_active_school_distance_m` | Primary numeric candidate. Retain raw metres in the feature table; if a log transform is tested, fit/apply it inside each cross-validation fold. |
| `education_nearest_school_sector` | Interpretation and pre-declared sensitivity candidate only. It has a plausible contextual meaning but the nearest school's governance does not directly measure pedestrian exposure. |
| `education_nearest_school_low_year`, `education_nearest_school_high_year` | Interpretation and sensitivity candidates only. Preserve official codes; do not impose an ordinal “education quality” score. |
| `education_nearest_school_code`, `education_nearest_school_name` | Audit identifiers only; never predictors. |
| `education_data_status` | Retrieval/match QA only; never a substantive predictor. |

`TotalSchool`, `TotalPrimaryK_6` and `TotalSecondary7_12` are excluded because the snapshot contains zero totals and enrolment is not comparable across every sector, year range and school type. A nearest school's enrolment also does not establish demand at the surveyed road. `RemCode`/`RemoteArea`, education region, address and postcode are excluded because they describe the school or administration, not the model site; site remoteness should come from the team's dedicated geographic feature source.

In [4]:
PROJECT_BOUNDS = {
    "minimum_longitude": 112.0,
    "maximum_longitude": 129.0,
    "minimum_latitude": -36.0,
    "maximum_latitude": -13.0,
}


def inside_project_bounds(frame: pd.DataFrame) -> pd.Series:
    # Return rows inside the declared mainland-WA project extent.
    return (
        frame["longitude"].between(
            PROJECT_BOUNDS["minimum_longitude"],
            PROJECT_BOUNDS["maximum_longitude"],
        )
        & frame["latitude"].between(
            PROJECT_BOUNDS["minimum_latitude"],
            PROJECT_BOUNDS["maximum_latitude"],
        )
    )


site_scope_frame = sites.rename(columns={"Latitude": "latitude", "Longitude": "longitude"})
if not inside_project_bounds(site_scope_frame).all():
    raise ValueError("At least one model site is outside the declared project extent.")

school_scope_frame = schools.rename(columns={"Latitude": "latitude", "Longitude": "longitude"})
school_scope_mask = inside_project_bounds(school_scope_frame)
outside_scope_schools = schools.loc[
    ~school_scope_mask,
    ["Code", "SchoolName", "PhysicalTown", "Latitude", "Longitude"],
].copy()
wa_schools = schools.loc[school_scope_mask].copy().reset_index(drop=True)

expected_outside_codes = {"4152", "4157"}
if set(outside_scope_schools["Code"].astype(str)) != expected_outside_codes:
    raise ValueError(
        "The records outside the project extent changed; review the scope before continuing."
    )
if wa_schools.empty:
    raise ValueError("No schools remain after applying the project extent.")

print(f"Schools inside project extent: {len(wa_schools):,}")
print(f"Schools outside project extent: {len(outside_scope_schools):,}")
print(outside_scope_schools.to_string(index=False))

Schools inside project extent: 1,150
Schools outside project extent: 2
Code                            SchoolName     PhysicalTown  Latitude  Longitude
4152 CHRISTMAS ISLAND DISTRICT HIGH SCHOOL CHRISTMAS ISLAND  -10.4352   105.6686
4157    COCOS ISLANDS DISTRICT HIGH SCHOOL     COCOS ISLAND  -12.1880    96.8288


In [5]:
def local_utm_epsg(latitude: float, longitude: float) -> int:
    # Return the WGS84 UTM EPSG code for one valid site coordinate.
    zone = int((longitude + 180.0) // 6.0) + 1
    zone = min(max(zone, 1), 60)
    return (32600 if latitude >= 0 else 32700) + zone


def nearest_school_for_site(site: pd.Series, school_frame: pd.DataFrame) -> dict:
    # Find the nearest in-scope school after projecting to the site's local UTM CRS.
    latitude = float(site["latitude"])
    longitude = float(site["longitude"])
    if not (-90 <= latitude <= 90 and -180 <= longitude <= 180):
        return {
            "site_id": site["site_id"],
            "education_data_status": "invalid_coordinate",
        }

    epsg = local_utm_epsg(latitude, longitude)
    transformer = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)
    site_x, site_y = transformer.transform(longitude, latitude)
    school_x, school_y = transformer.transform(
        school_frame["Longitude"].to_numpy(dtype=float),
        school_frame["Latitude"].to_numpy(dtype=float),
    )
    distances = np.hypot(school_x - site_x, school_y - site_y)
    if not np.isfinite(distances).all():
        raise ValueError(f"Projection returned a non-finite school distance for {site['site_id']}.")

    nearest_position = int(np.argmin(distances))
    nearest = school_frame.iloc[nearest_position]
    return {
        "site_id": site["site_id"],
        "education_nearest_active_school_distance_m": float(distances[nearest_position]),
        "education_nearest_school_sector": nearest["Sector"],
        "education_nearest_school_low_year": nearest["LowYear"],
        "education_nearest_school_high_year": nearest["HighYear"],
        "education_nearest_school_code": nearest["Code"],
        "education_nearest_school_name": nearest["SchoolName"],
        "education_data_status": "matched",
        "_distance_epsg": epsg,
        "_nearest_school_latitude": float(nearest["Latitude"]),
        "_nearest_school_longitude": float(nearest["Longitude"]),
    }


match_rows = [nearest_school_for_site(site, wa_schools) for _, site in sites.iterrows()]
matches = pd.DataFrame(match_rows)

feature_columns = [
    "site_id",
    "education_nearest_active_school_distance_m",
    "education_nearest_school_sector",
    "education_nearest_school_low_year",
    "education_nearest_school_high_year",
    "education_nearest_school_code",
    "education_nearest_school_name",
    "education_data_status",
]
features = matches[feature_columns].copy()
features["site_id"] = features["site_id"].astype("string")
features["education_nearest_school_code"] = features[
    "education_nearest_school_code"
].astype("string")

if len(features) != len(sites):
    raise ValueError("The feature output row count does not match the site input.")
if features["site_id"].isna().any() or features["site_id"].duplicated().any():
    raise ValueError("The feature output must contain one unique row per site_id.")
if set(features["site_id"]) != set(sites["site_id"]):
    raise ValueError("The feature output site_id coverage differs from Model_Data.")
if not features["education_data_status"].eq("matched").all():
    raise ValueError("At least one current model site did not match an active school.")
if features.drop(columns="site_id").isna().any().any():
    raise ValueError("The current matched feature output unexpectedly contains missing values.")

forbidden_fragments = ("obs_peak", "latitude", "longitude", "qa_status")
if any(
    fragment in column
    for column in features.columns
    for fragment in forbidden_fragments
):
    raise ValueError("The source-specific feature table contains a forbidden base-table field.")

print(features.head().to_string(index=False))

  site_id  education_nearest_active_school_distance_m education_nearest_school_sector education_nearest_school_low_year education_nearest_school_high_year education_nearest_school_code       education_nearest_school_name education_data_status
250303-01                                20481.040474                  NON-GOVERNMENT                               KIN                                Y06                          1188                  ST JOSEPH'S SCHOOL               matched
250303-02                                 1349.539551                      GOVERNMENT                               KIN                                Y06                          5760       DJIDI DJIDI ABORIGINAL SCHOOL               matched
250303-03                                  318.566350                      GOVERNMENT                               Y07                                Y12                          4014           COLLIE SENIOR HIGH SCHOOL               matched
250303-04                   

## 4. Save the independent feature table and verify the team merge

The workbook contains one flat sheet, with `site_id` first and only education-source fields after it. No model target, coordinate, road field or permanent train/test assignment is included. School code remains text because it is an identifier, while distance remains a numeric value in metres.

The dry-run merge below uses `validate="one_to_one"`, matching the team's planned combination pattern. It proves that the feature table can be joined without changing the 49-row foundation table. The merged frame is not saved here; training code should assemble all team feature sources and then filter `qa_status == "ready"`.

In [6]:
merge_check = model_data.merge(
    features,
    on="site_id",
    how="left",
    validate="one_to_one",
)
if len(merge_check) != len(model_data):
    raise ValueError("The education merge changed the Model_Data row count.")
if not merge_check["education_data_status"].eq("matched").all():
    raise ValueError("At least one Model_Data row lacks a matched education feature.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
features.to_excel(
    OUTPUT_PATH,
    sheet_name="WA_Education_Features",
    index=False,
)

roundtrip_workbook = pd.ExcelFile(OUTPUT_PATH)
if roundtrip_workbook.sheet_names != ["WA_Education_Features"]:
    raise ValueError("The exported feature workbook has unexpected sheets.")
roundtrip = pd.read_excel(
    OUTPUT_PATH,
    sheet_name="WA_Education_Features",
    dtype={"site_id": "string", "education_nearest_school_code": "string"},
)
if list(roundtrip.columns) != feature_columns:
    raise ValueError("The exported workbook column order changed during round-trip.")
if len(roundtrip) != len(features) or set(roundtrip["site_id"]) != set(features["site_id"]):
    raise ValueError("The exported workbook failed the row or site_id round-trip check.")

print(f"Saved: {OUTPUT_PATH}")
print(f"Exported rows: {len(roundtrip)}")
print(f"Exported columns: {len(roundtrip.columns)}")

Saved: /workspace/outputs/02_imagery_features/wa-education-schools/haoqi-05-wa-education-features.xlsx
Exported rows: 49
Exported columns: 8


## 5. Quality checks and spatial spot checks

The checks below report source coverage, status, distance distribution, nearest-school category values and the number of distinct nearest schools. The distance distribution matters because a strongly right-skewed variable may need a fold-safe transform or a robust model; it is not a reason to delete a valid remote site.

Four site IDs were fixed before inspecting the pedestrian targets to cover major-city, inner-regional, remote and very-remote contexts and both observed nearest-school sectors. Each panel projects schools within 30 km into the site's local UTM CRS, marks the survey site with an X and the matched school with a star, and draws the selected connection. This is a linkage QA plot, not a road-network or exposure map.

In [7]:
distance_summary = features[
    "education_nearest_active_school_distance_m"
].describe(percentiles=[0.25, 0.5, 0.75]).round(1)

print("Match status:")
print(features["education_data_status"].value_counts(dropna=False).to_string())
print("\nNearest-school distance in metres:")
print(distance_summary.to_string())
print("\nNearest-school sector:")
print(features["education_nearest_school_sector"].value_counts().to_string())
print("\nNearest-school low-year code:")
print(features["education_nearest_school_low_year"].value_counts().to_string())
print("\nNearest-school high-year code:")
print(features["education_nearest_school_high_year"].value_counts().to_string())
print(
    "\nDistinct nearest schools:",
    features["education_nearest_school_code"].nunique(),
)

spot_check_ids = ["250303-15", "250303-10", "250341-05", "250341-02"]
spot_checks = (
    sites[["site_id", "latitude", "longitude"]]
    .merge(matches, on="site_id", how="inner", validate="one_to_one")
    .loc[lambda frame: frame["site_id"].isin(spot_check_ids)]
    .copy()
)
spot_checks["site_id"] = pd.Categorical(
    spot_checks["site_id"], categories=spot_check_ids, ordered=True
)
spot_checks = spot_checks.sort_values("site_id")
print("\nPre-declared spot checks:")
print(
    spot_checks[
        [
            "site_id",
            "education_nearest_school_name",
            "education_nearest_school_sector",
            "education_nearest_active_school_distance_m",
            "_distance_epsg",
        ]
    ].to_string(index=False)
)

Match status:
education_data_status
matched    49

Nearest-school distance in metres:
count       49.0
mean      1075.4
std       2867.1
min         58.9
25%        340.7
50%        514.6
75%        920.4
max      20481.0

Nearest-school sector:
education_nearest_school_sector
GOVERNMENT        31
NON-GOVERNMENT    18

Nearest-school low-year code:
education_nearest_school_low_year
KIN    35
PKG     8
Y07     4
Y11     1
Y10     1

Nearest-school high-year code:
education_nearest_school_high_year
Y06    32
Y12    13
Y10     4

Distinct nearest schools: 46

Pre-declared spot checks:
  site_id education_nearest_school_name education_nearest_school_sector  education_nearest_active_school_distance_m  _distance_epsg
250303-15    GREENFIELDS PRIMARY SCHOOL                      GOVERNMENT                                  964.017512           32750
250303-10            ST BRIGID'S SCHOOL                  NON-GOVERNMENT                                  373.590662           32750
250341-05   BAY